# <a name='Import_Libraries_and_Data'></a>

<div style="border-radius:10px;
            background-color:#ffffff;
            border-style: solid;
            border-color: #0b0265;
            letter-spacing:0.5px;">

<center><h3 style="padding: 5px 0px; color:#0b0265; font-weight: bold; font-family: Cursive">
1. Import Libraries and Data</h3></center>
</div>

In [1]:
import numpy as np
import pandas as pd


import matplotlib.pyplot as plt 
import seaborn as sns


import ast

import re
import string

import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download("omw-1.4")

from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.neighbors import NearestNeighbors




import warnings
warnings.filterwarnings('ignore')


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\darkl\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\darkl\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\darkl\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [ ]:
movies = pd.read_csv("movies_metadata.csv", low_memory=False)

credit = pd.read_csv("credits.csv")

keywords = pd.read_csv("keywords.csv")

# <a name='Data processing and cleaning'></a>

<div style="border-radius:10px;
            background-color:#ffffff;
            border-style: solid;
            border-color: #0b0265;
            letter-spacing:0.5px;">

<center><h3 style="padding: 5px 0px; color:#0b0265; font-weight: bold; font-family: Cursive">
2. Data processing and cleaning</h3></center>
</div>

converting all ids to int

In [ ]:
movies = movies[pd.to_numeric(movies["id"], errors="coerce").notnull()]
movies["id"] = movies["id"].astype(int)

credit["id"] = credit["id"].astype(int)

keywords["id"] = keywords["id"].astype(int)

merging three dataframes


In [ ]:
movies = movies.merge(credit, on="id", how="inner")
movies = movies.merge(keywords, on="id", how="inner")

In [ ]:
print(movies.shape)

(46628, 27)


In [ ]:
movies = movies[
    [
        "id",
        "title",
        "overview",
        "genres",
        "keywords",
        "cast",
        "crew",
        "tagline",
        "vote_average",
        "vote_count",
        "popularity",
        "release_date",
        "runtime",
    ]
]

removing duplicated rows

In [ ]:
movies.duplicated().sum()

np.int64(1172)

In [ ]:
movies.drop_duplicates(inplace=True)
movies.reset_index(drop=True, inplace=True)
movies.duplicated().sum()


np.int64(0)

handle null values


In [ ]:
movies.isnull().sum()

id                  0
title               3
overview          954
genres              0
keywords            0
cast                0
crew                0
tagline         25046
vote_average        3
vote_count          3
popularity          3
release_date       87
runtime           260
dtype: int64

In [ ]:
movies=movies.dropna(subset=['title'])
movies['overview']=movies['overview'].fillna('')
movies['tagline']=movies['tagline'].fillna('')

# <a name='Feature Enginnering'></a>

<div style="border-radius:10px;
            background-color:#ffffff;
            border-style: solid;
            border-color: #0b0265;
            letter-spacing:0.5px;">

<center><h3 style="padding: 5px 0px; color:#0b0265; font-weight: bold; font-family: Cursive">
3. Feature Enginnering</h3></center>
</div>

extracting genres,keyword,cast,director

In [ ]:
movies['genres']=movies['genres'].apply(lambda x: " ".join([i['name'] for i in ast.literal_eval(x)]))
movies["keywords"] = movies["keywords"].apply(lambda x: " ".join([i["name"] for i in ast.literal_eval(x)]))

In [ ]:
def extract_cast(text):
    cast_list = []

    for actor in ast.literal_eval(text)[:3]:
        cast_list.append(actor["name"].replace(" ", ""))

    return " ".join(cast_list)
movies["cast"] = movies["cast"].apply(extract_cast)

In [ ]:
def extract_director(text):
    for member in ast.literal_eval(text):
        if member["job"] == "Director":
            return member["name"].replace(" ", "")

    return ""
movies["crew"] = movies["crew"].apply(extract_director)

In [ ]:
movies.head()

,id,title,overview,genres,keywords,cast,crew,tagline,vote_average,vote_count,popularity,release_date,runtime
0,862,Toy Story,"Led by Woody, Andy's toys live happily in his ...",Animation Comedy Family,jealousy toy boy friendship friends rivalry bo...,TomHanks TimAllen DonRickles,JohnLasseter,,7.7,5415.0,21.946943,1995-10-30,81.0
1,8844,Jumanji,When siblings Judy and Peter discover an encha...,Adventure Fantasy Family,board game disappearance based on children's b...,RobinWilliams JonathanHyde KirstenDunst,JoeJohnston,Roll the dice and unleash the excitement!,6.9,2413.0,17.015539,1995-12-15,104.0
2,15602,Grumpier Old Men,A family wedding reignites the ancient feud be...,Romance Comedy,fishing best friend duringcreditsstinger old men,WalterMatthau JackLemmon Ann-Margret,HowardDeutch,Still Yelling. Still Fighting. Still Ready for...,6.5,92.0,11.7129,1995-12-22,101.0
3,31357,Waiting to Exhale,"Cheated on, mistreated and stepped on, the wom...",Comedy Drama Romance,based on novel interracial relationship single...,WhitneyHouston AngelaBassett LorettaDevine,ForestWhitaker,Friends are the people who let you be yourself...,6.1,34.0,3.859495,1995-12-22,127.0
4,11862,Father of the Bride Part II,Just when George Banks has recovered from his ...,Comedy,baby midlife crisis confidence aging daughter ...,SteveMartin DianeKeaton MartinShort,CharlesShyer,Just When His World Is Back To Normal... He's ...,5.7,173.0,8.387519,1995-02-10,106.0


In [ ]:
movies["tag"] = (
movies["overview"] + " " +
    movies["genres"] + " " +
    movies["keywords"] + " " +
    movies["cast"] + " " +
    movies["crew"] + " " +
    movies["tagline"]
)

In [ ]:
movies['tag'][0]

"Led by Woody, Andy's toys live happily in his room until Andy's birthday brings Buzz Lightyear onto the scene. Afraid of losing his place in Andy's heart, Woody plots against Buzz. But when circumstances separate Buzz and Woody from their owner, the duo eventually learns to put aside their differences. Animation Comedy Family jealousy toy boy friendship friends rivalry boy next door new toy toy comes to life TomHanks TimAllen DonRickles JohnLasseter "

In [ ]:
final_df = movies[
    [
        "id",
        "title",
        "tag",
        "vote_average",
        "vote_count",
        "popularity",
        "release_date",
        "runtime"
    ]
].copy()
final_df["vote_average"] = pd.to_numeric(final_df["vote_average"], errors="coerce")
final_df["vote_count"] = pd.to_numeric(final_df["vote_count"], errors="coerce")
final_df["popularity"] = pd.to_numeric(final_df["popularity"], errors="coerce")
final_df["runtime"] = pd.to_numeric(final_df["runtime"], errors="coerce")

# <a name='NLP text preprocessing'></a>

<div style="border-radius:10px;
            background-color:#ffffff;
            border-style: solid;
            border-color: #0b0265;
            letter-spacing:0.5px;">

<center><h3 style="padding: 5px 0px; color:#0b0265; font-weight: bold; font-family: Cursive">
3. Text preprocessing</h3></center>
</div>

In [ ]:
def clean_text(text):
    # Convert text to string
    text = str(text)
    #convert to lower case
    text = text.lower()

    # Remove punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))

    # Remove numbers
    text = re.sub(r'\d+', '', text)

    # Remove special symbols
    text = re.sub(r'[^a-zA-Z\s]', '', text)

    # Remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()

    words=text.split()
    stop_words=set(stopwords.words('english'))
    #remove stop_words
    words=[word for word in words if word not in stop_words]

    #lemmetizer
    lemmetizer=WordNetLemmatizer()
    words=[lemmetizer.lemmatize(word) for word in words]
    return " ". join(words)

In [ ]:
movies['tag']=movies['tag'].apply(clean_text)

# <a name='Vectorization'></a>

<div style="border-radius:10px;
            background-color:#ffffff;
            border-style: solid;
            border-color: #0b0265;
            letter-spacing:0.5px;">

<center><h3 style="padding: 5px 0px; color:#0b0265; font-weight: bold; font-family: Cursive">
4. Vectorization</h3></center>
</div>

In [ ]:
tfidf = TfidfVectorizer(
    max_features=50000,
    stop_words="english",
    ngram_range=(1,2),
    min_df=2,
    max_df=0.8,
    sublinear_tf=True
)
tfidf_matrix=tfidf.fit_transform(movies['tag'])


# <a name='nearest neighbor'></a>

<div style="border-radius:10px;
            background-color:#ffffff;
            border-style: solid;
            border-color: #0b0265;
            letter-spacing:0.5px;">

<center><h3 style="padding: 5px 0px; color:#0b0265; font-weight: bold; font-family: Cursive">
5. Nearest_Neighbor Model </h3></center>
</div>

In [ ]:
nn_model = NearestNeighbors(
    n_neighbors=11,
    metric="cosine",
    algorithm="brute"
)
nn_model.fit(tfidf_matrix)

,"n_neighbors n_neighbors: int, default=5Number of neighbors to use by default for :meth:`kneighbors` queries.",11
,"algorithm algorithm: {'auto', 'ball_tree', 'kd_tree', 'brute'}, default='auto'Algorithm used to compute the nearest neighbors:- 'ball_tree' will use :class:`BallTree`- 'kd_tree' will use :class:`KDTree`- 'brute' will use a brute-force search.- 'auto' will attempt to decide the most appropriate algorithm based on the values passed to :meth:`fit` method.Note: fitting on sparse input will override the setting ofthis parameter, using brute force.",'brute'
,"metric metric: str or callable, default='minkowski'Metric to use for distance computation. Default is ""minkowski"", whichresults in the standard Euclidean distance when p = 2. See thedocumentation of `scipy.spatial.distance<https://docs.scipy.org/doc/scipy/reference/spatial.distance.html>`_ andthe metrics listed in:class:`~sklearn.metrics.pairwise.distance_metrics` for valid metricvalues.If metric is ""precomputed"", X is assumed to be a distance matrix andmust be square during fit. X may be a :term:`sparse graph`, in whichcase only ""nonzero"" elements may be considered neighbors.If metric is a callable function, it takes two arrays representing 1Dvectors as inputs and must return one value indicating the distancebetween those vectors. This works for Scipy's metrics, but is lessefficient than passing the metric name as a string.",'cosine'
,"radius radius: float, default=1.0Range of parameter space to use by default for :meth:`radius_neighbors`queries.",1.0
,"leaf_size leaf_size: int, default=30Leaf size passed to BallTree or KDTree. This can affect thespeed of the construction and query, as well as the memoryrequired to store the tree. The optimal value depends on thenature of the problem.",30
,"p p: float (positive), default=2Parameter for the Minkowski metric fromsklearn.metrics.pairwise.pairwise_distances. When p = 1, this isequivalent to using manhattan_distance (l1), and euclidean_distance(l2) for p = 2. For arbitrary p, minkowski_distance (l_p) is used.",2
,"metric_params metric_params: dict, default=NoneAdditional keyword arguments for the metric function.",None
,"n_jobs n_jobs: int, default=NoneThe number of parallel jobs to run for neighbors search.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
Name,Type,Value
effective_metric_ effective_metric_: strMetric used to compute distances to neighbors.,str,'cosine'
effective_metric_params_ effective_metric_params_: dictParameters for the metric used to compute distances to neighbors.,dict,{}


# <a name='indicies and recommendation function'></a>

<div style="border-radius:10px;
            background-color:#ffffff;
            border-style: solid;
            border-color: #0b0265;
            letter-spacing:0.5px;">

<center><h3 style="padding: 5px 0px; color:#0b0265; font-weight: bold; font-family: Cursive">
6. Indices and Recommendation Function </h3></center>
</div>

In [ ]:
final_df["title_lower"] = final_df["title"].str.lower()
indices = pd.Series(final_df.index, index=final_df["title_lower"])
indices = indices[~indices.index.duplicated(keep="first")]

In [ ]:
indices["Interstellar"]

np.int64(22876)

In [ ]:
C = final_df["vote_average"].mean()

m = final_df["vote_count"].quantile(0.70)

In [ ]:
def recommend(movie_name, n_recommendations=10):

    # -----------------------------
    # Convert input to lowercase
    # -----------------------------
    movie_name = movie_name.lower()

    # -----------------------------
    # Check if movie exists
    # -----------------------------
    if movie_name not in indices:
        return "Movie not found!"

    # -----------------------------
    # Get movie index
    # -----------------------------
    movie_idx = indices[movie_name]

    # -----------------------------
    # Find nearest neighbours
    # -----------------------------
    distances, neighbour_indices = nn_model.kneighbors(
        tfidf_matrix[movie_idx],
        n_neighbors=n_recommendations + 1
    )

    # -----------------------------
    # Remove queried movie
    # -----------------------------
    distances = distances.flatten()[1:]
    neighbour_indices = neighbour_indices.flatten()[1:]

    # -----------------------------
    # Convert cosine distance to similarity
    # -----------------------------
    similarity = 1 - distances

    # -----------------------------
    # Get movie details
    # -----------------------------
    recommendations = final_df.iloc[neighbour_indices][
        [
            "title",
            "vote_average",
            "vote_count",
            "popularity",
            "release_date"
        ]
    ].copy()

    recommendations["similarity"] = similarity

    # -----------------------------
    # IMDb Weighted Rating
    # -----------------------------
    recommendations["weighted_rating"] = (
        (
            recommendations["vote_count"] /
            (recommendations["vote_count"] + m)
        )
        * recommendations["vote_average"]
    ) + (
        (
            m /
            (recommendations["vote_count"] + m)
        )
        * C
    )

    # -----------------------------
    # Normalize weighted rating
    # -----------------------------
    recommendations["rating_score"] = (
        recommendations["weighted_rating"] /
        recommendations["weighted_rating"].max()
    )

    # -----------------------------
    # Normalize popularity
    # -----------------------------
    recommendations["popularity_score"] = (
        recommendations["popularity"] /
        recommendations["popularity"].max()
    )

    # -----------------------------
    # Final recommendation score
    # -----------------------------
    recommendations["final_score"] = (
        0.85 * recommendations["similarity"] +
        0.10 * recommendations["rating_score"] +
        0.05 * recommendations["popularity_score"]
    )

    # -----------------------------
    # Sort by final score
    # -----------------------------
    recommendations = recommendations.sort_values(
        by="final_score",
        ascending=False
    )

    # -----------------------------
    # Return results
    # -----------------------------
    return recommendations[
        [
            "title",
            "similarity",
            "weighted_rating",
            "vote_average",
            "vote_count",
            "popularity",
            "final_score"
        ]
    ].reset_index(drop=True)


In [ ]:
recommend("The Dark Knight")

,title,similarity,weighted_rating,vote_average,vote_count,popularity,final_score
0,Batman Begins,0.405910,7.493757,7.5,7511.0,28.505341,0.489033
1,The Dark Knight Rises,0.408340,7.594665,7.6,9263.0,20.582580,0.479826
2,Batman: Under the Red Hood,0.350624,7.497629,7.6,459.0,7.039325,0.407949
3,Batman Returns,0.289586,6.585819,6.6,1706.0,15.001681,0.356725
4,Batman,0.269353,6.984079,7.0,2145.0,19.106730,0.351300
5,Batman Forever,0.304434,5.206726,5.2,1529.0,13.321354,0.348515
6,Batman Unlimited: Monster Mayhem,0.307216,5.853115,6.0,40.0,2.772324,0.342611
7,Batman v Superman: Dawn of Justice,0.238528,5.699716,5.7,7189.0,31.435879,0.327798
8,Batman: The Killing Joke,0.249572,6.171475,6.2,485.0,6.488277,0.303716
9,Batman: Return of the Caped Crusaders,0.239837,6.562137,7.2,37.0,3.495883,0.295826


In [ ]:
recommend("Avenger")

,title,similarity,weighted_rating,vote_count,popularity,final_score
0,Disaster L.A.: The Last Zombie Apocalypse Begi...,0.290935,4.927697,13.0,1.320585,0.362103
1,Zombie Resurrection,0.262415,5.382339,6.0,0.848918,0.336275
2,Devil's Playground,0.223498,5.371352,36.0,2.436746,0.335581
3,The Book of Fate,0.265745,5.461204,2.0,0.203204,0.327260
4,Plague,0.243638,5.188351,22.0,1.125723,0.322542
5,Cyborg X,0.212595,5.269803,13.0,1.494695,0.305176
6,Night of the Living Dead: Darkest Dawn,0.207237,5.537177,6.0,1.112812,0.297545
7,Darkest Day,0.210081,4.898813,15.0,1.239071,0.291191
8,Chrysalis,0.207519,5.461016,7.0,0.537234,0.284618
9,Survivor,0.204701,5.618100,0.0,0.002603,0.274049


In [ ]:
recommend("Interstellar")

,title,similarity,weighted_rating,vote_average,vote_count,popularity,final_score
0,Love Finds You in Charm,0.229943,5.653306,5.8,6.0,0.276956,0.278528
1,Witness,0.141628,6.916753,7.0,390.0,10.312017,0.270383
2,Love Finds You In Sugarcreek,0.198733,5.808707,7.0,4.0,0.712788,0.256359
3,Deadly Blessing,0.178880,5.271096,5.0,32.0,3.175395,0.243652
4,Devil's Playground,0.169118,6.006474,6.7,14.0,1.078421,0.235819
5,Expecting Amish,0.158059,5.679766,5.9,7.0,1.031839,0.221470
6,Secrets of Eden,0.144152,5.459758,4.8,6.0,3.573543,0.218792
7,Hell,0.140671,5.812929,6.3,10.0,2.142094,0.213998
8,Birch Interval,0.133714,5.786635,10.0,1.0,0.320041,0.198870
9,Die zwölf Geschworenen,0.133473,5.748173,9.0,1.0,0.074898,0.196920
